# 53 — LSEG prospective accumulation population gate

**Candidate RQ:** strategy construction and risk allocation. Does the frozen
sparse cross-scorer event rule retain next-open value on genuinely new dates?

This notebook audits the first immutable post-cutoff LSEG headline batch and
applies the **population gate before scorer execution or return loading**. The
batch interval was closed and committed before retrieval. It is not pooled
with FNSPID or the opened LSEG window, and licensed headline text is never
displayed or written to notebook outputs.

In [ ]:
from __future__ import annotations

import hashlib
import json
import subprocess
import sys
from pathlib import Path

import exchange_calendars as xcals
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "final_experiments":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from final_experiments.lib.plots import CATEGORICAL, INK, apply_house_style  # noqa: E402
from sentiment_benchmark.lseg_source import load_lseg_collection_config  # noqa: E402

PARENT_SPEC_PATH = (
    REPO_ROOT
    / "final_experiments/frozen_specs/lseg_gemma_finbert_hybrid_prospective_v1.json"
)
ACQUISITION_SPEC_PATH = (
    REPO_ROOT
    / "final_experiments/frozen_specs/lseg_gemma_finbert_hybrid_prospective_acquisition_batch_20260806.json"
)
CONFIG_PATH = (
    REPO_ROOT
    / "configs/lseg_us_sector_33_prospective_20260626_20260806_headlines.toml"
)
RAW_DIR = (
    REPO_ROOT
    / "Data/collections/lseg_us_sector_33_prospective_20260626_20260806_headlines/raw"
    / "lseg_us_sector_33_prospective_20260626_20260806_headlines"
)
RAW_MANIFEST_PATH = RAW_DIR / "manifest.json"
HEADLINES_PATH = RAW_DIR / "headlines.jsonl"
OUTPUT_DIR = REPO_ROOT / "final_experiments/outputs/53_lseg_prospective_population_gate"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

apply_house_style()
pd.set_option("display.max_columns", 100)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def utc_timestamp(value: object) -> pd.Timestamp:
    timestamp = pd.Timestamp(value)
    if timestamp.tzinfo is None:
        return timestamp.tz_localize("UTC")
    return timestamp.tz_convert("UTC")


def git_head() -> str:
    return subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, text=True
    ).strip()

## Frozen acquisition and collection integrity

The source-control hash and the collector's parsed-config hash must both match
the pre-retrieval record. A completed collector manifest is required. Safe
repeated-cursor terminal stops may be reported, but unresolved pagination or
story-body retrieval fails closed.

In [ ]:
parent_spec = json.loads(PARENT_SPEC_PATH.read_text(encoding="utf-8"))
acquisition_spec = json.loads(ACQUISITION_SPEC_PATH.read_text(encoding="utf-8"))
raw_manifest = json.loads(RAW_MANIFEST_PATH.read_text(encoding="utf-8"))
config = load_lseg_collection_config(CONFIG_PATH)

assert parent_spec["status"] == "frozen_awaiting_genuinely_new_lseg_dates"
assert acquisition_spec["status"] == "frozen_before_retrieval"
assert sha256_file(CONFIG_PATH) == acquisition_spec["collection_config"]["file_sha256"]
assert config.config_sha256 == acquisition_spec["collection_config"]["parsed_config_sha256"]
assert raw_manifest["config_sha256"] == config.config_sha256
assert raw_manifest["status"] == "completed"
assert raw_manifest["config"]["collection"]["id"] == config.collection_id
assert raw_manifest["counts"]["fetch_story_bodies"] is False
assert raw_manifest["counts"]["stories"] == 0
assert raw_manifest["counts"]["failed_stories"] == 0
assert sha256_file(HEADLINES_PATH) == raw_manifest["files"]["headlines_jsonl"]["sha256"]

safe_anomaly_reasons = {"repeated_cursor_duplicate_page"}
pagination_anomalies = raw_manifest.get("pagination_anomalies", [])
assert all(row.get("reason") in safe_anomaly_reasons for row in pagination_anomalies)

display(
    pd.DataFrame(
        [
            {
                "collection_id": config.collection_id,
                "status": raw_manifest["status"],
                "companies": len(config.companies),
                "requested_start": config.start,
                "requested_end_exclusive": config.end,
                "headline_pages": raw_manifest["counts"]["headline_pages"],
                "deduplicated_headlines": raw_manifest["counts"]["headlines"],
                "story_requests": raw_manifest["counts"]["stories"],
                "safe_cursor_stops": len(pagination_anomalies),
            }
        ]
    )
)

## Metadata-only coverage audit

The scan retains only timestamps, story identifiers, and matched symbols in
memory. It neither displays nor exports headline text. The shared cutoff
boundary is permitted for retrieval completeness; only timestamps strictly
after it are prospective-eligible.

In [ ]:
cutoff = utc_timestamp(parent_spec["claim_boundary"]["prospective_start_utc"])
requested_end = utc_timestamp(config.end)
first_created_values: list[pd.Timestamp] = []
version_created_values: list[pd.Timestamp] = []
matched_symbols: set[str] = set()
story_ids: set[str] = set()
duplicate_story_ids = 0
missing_timestamp_rows = 0
eligible_headlines = 0
boundary_or_earlier_headlines = 0
headline_rows = 0

with HEADLINES_PATH.open("r", encoding="utf-8") as handle:
    for line in handle:
        row = json.loads(line)
        headline_rows += 1
        story_id = str(row.get("story_id") or "")
        if not story_id:
            raise ValueError("headline row is missing story_id")
        if story_id in story_ids:
            duplicate_story_ids += 1
        story_ids.add(story_id)
        matched_symbols.update(str(value) for value in row.get("matched_symbols", []))
        first_created = row.get("first_created")
        version_created = row.get("version_created")
        if first_created:
            first_created_values.append(utc_timestamp(first_created))
        if version_created:
            version_created_values.append(utc_timestamp(version_created))
        event_timestamp = first_created or version_created
        if not event_timestamp:
            missing_timestamp_rows += 1
            continue
        if utc_timestamp(event_timestamp) > cutoff:
            eligible_headlines += 1
        else:
            boundary_or_earlier_headlines += 1

expected_symbols = {
    symbol
    for members in parent_spec["universe"]["sectors"].values()
    for symbol in members
}
assert headline_rows == raw_manifest["counts"]["headlines"]
assert duplicate_story_ids == 0
assert matched_symbols == expected_symbols
assert missing_timestamp_rows == 0
assert eligible_headlines + boundary_or_earlier_headlines == headline_rows
assert max(first_created_values) < requested_end

coverage_audit = pd.DataFrame(
    [
        {
            "collection_id": config.collection_id,
            "headline_rows": headline_rows,
            "unique_story_ids": len(story_ids),
            "matched_symbols": len(matched_symbols),
            "first_first_created_utc": min(first_created_values).isoformat(),
            "last_first_created_utc": max(first_created_values).isoformat(),
            "first_version_created_utc": min(version_created_values).isoformat(),
            "last_version_created_utc": max(version_created_values).isoformat(),
            "strictly_post_cutoff_headlines": eligible_headlines,
            "boundary_or_earlier_headlines": boundary_or_earlier_headlines,
            "missing_timestamp_rows": missing_timestamp_rows,
            "duplicate_story_ids": duplicate_story_ids,
            "licensed_headline_text_exported": False,
        }
    ]
)
display(coverage_audit)

## Frozen minimum-population gate

The calendar count is an **upper bound**, not a price-completeness claim: no
price file or return is loaded. If even this upper bound is below 120 XNYS
sessions, active-session gates are not evaluated and both local FinBERT and
paid external Gemma scoring are deferred.

In [ ]:
calendar = xcals.get_calendar("XNYS", start=cutoff.date(), end="2027-03-01")
all_sessions = calendar.sessions_in_range(cutoff.date(), "2027-03-01")
eligible_sessions = all_sessions[
    (all_sessions > cutoff.tz_localize(None).normalize())
    & (all_sessions < requested_end.tz_localize(None).normalize())
]
required_calendar_sessions = int(
    parent_spec["minimum_evaluation_population"]["complete_price_calendar_sessions"]
)
required_active_sessions = int(
    parent_spec["minimum_evaluation_population"]["primary_active_sessions"]
)
required_active_half = int(
    parent_spec["minimum_evaluation_population"][
        "active_sessions_each_chronological_half"
    ]
)
earliest_120th_session = all_sessions[all_sessions > cutoff.tz_localize(None).normalize()][
    required_calendar_sessions - 1
]
calendar_gate_pass = len(eligible_sessions) >= required_calendar_sessions
assert not calendar_gate_pass

gate_table = pd.DataFrame(
    [
        {
            "gate": "complete_price_calendar_sessions",
            "required": required_calendar_sessions,
            "observed_or_upper_bound": len(eligible_sessions),
            "status": "FAIL_BY_CALENDAR_UPPER_BOUND",
            "reason": "No price data loaded; closed interval cannot contain enough XNYS sessions.",
        },
        {
            "gate": "primary_active_sessions",
            "required": required_active_sessions,
            "observed_or_upper_bound": pd.NA,
            "status": "NOT_EVALUATED",
            "reason": "Calendar gate failed before scoring.",
        },
        {
            "gate": "active_sessions_each_chronological_half",
            "required": required_active_half,
            "observed_or_upper_bound": pd.NA,
            "status": "NOT_EVALUATED",
            "reason": "Calendar gate failed before scoring.",
        },
    ]
)
display(gate_table)
display(
    Markdown(
        f"**Stop:** the batch has at most **{len(eligible_sessions)}** post-cutoff "
        f"XNYS sessions versus **{required_calendar_sessions}** required. The earliest "
        f"possible 120th session is **{earliest_120th_session.date()}**. No scorer or "
        "return evaluation is launched."
    )
)

## Population-gate figure

In [ ]:
figure, axis = plt.subplots(figsize=(7.5, 4.2))
labels = ["Current batch\ncalendar upper bound", "Frozen minimum"]
values = [len(eligible_sessions), required_calendar_sessions]
bars = axis.bar(labels, values, color=[CATEGORICAL[0], INK["muted"]], width=0.56)
axis.set_ylabel("XNYS sessions")
axis.set_title("Prospective replay stops before scoring")
axis.axhline(required_calendar_sessions, color=INK["reference"], linestyle="--", linewidth=1)
for bar, value in zip(bars, values, strict=True):
    axis.text(
        bar.get_x() + bar.get_width() / 2,
        value + 3,
        f"{value}",
        ha="center",
        va="bottom",
        color=INK["primary"],
    )
axis.text(
    0.02,
    0.96,
    f"Earliest possible 120th session: {earliest_120th_session.date()}",
    transform=axis.transAxes,
    ha="left",
    va="top",
    color=INK["secondary"],
)
axis.set_ylim(0, required_calendar_sessions * 1.18)
figure.tight_layout()
figure_path = OUTPUT_DIR / "prospective_population_gate.png"
figure.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.show()

## Persisted licence-safe evidence

In [ ]:
coverage_path = OUTPUT_DIR / "collection_coverage_audit.csv"
gate_path = OUTPUT_DIR / "population_gate.csv"
source_hashes_path = OUTPUT_DIR / "source_hashes.csv"
manifest_path = OUTPUT_DIR / "manifest.json"

coverage_audit.to_csv(coverage_path, index=False, lineterminator="\n")
gate_table.to_csv(gate_path, index=False, lineterminator="\n")

source_paths = [
    PARENT_SPEC_PATH,
    ACQUISITION_SPEC_PATH,
    CONFIG_PATH,
    RAW_MANIFEST_PATH,
    HEADLINES_PATH,
]
source_hashes = pd.DataFrame(
    [
        {
            "path": str(path.relative_to(REPO_ROOT)),
            "sha256": sha256_file(path),
        }
        for path in source_paths
    ]
)
source_hashes.to_csv(source_hashes_path, index=False, lineterminator="\n")

manifest = {
    "schema_version": 1,
    "status": "stopped_before_scoring_and_returns",
    "notebook": "53_lseg_prospective_population_gate.ipynb",
    "git_commit_at_execution": git_head(),
    "candidate_rq": parent_spec["candidate_rq"],
    "collection": {
        "id": config.collection_id,
        "completed_at": raw_manifest["completed_at"],
        "headline_rows": headline_rows,
        "strictly_post_cutoff_headlines": eligible_headlines,
        "matched_symbols": len(matched_symbols),
        "story_bodies_requested": False,
        "safe_cursor_stops": len(pagination_anomalies),
    },
    "population_gate": {
        "post_cutoff_xnys_session_upper_bound": len(eligible_sessions),
        "required_complete_price_calendar_sessions": required_calendar_sessions,
        "earliest_possible_120th_session": str(earliest_120th_session.date()),
        "calendar_gate_pass": False,
        "active_session_gates_evaluated": False,
    },
    "stop_controls": {
        "finbert_scoring_launched": False,
        "paid_external_gemma_scoring_launched": False,
        "price_data_loaded": False,
        "returns_loaded": False,
        "performance_inference_run": False,
        "fnspid_pooling": False,
    },
    "sharing": {
        "contains_licensed_headline_text": False,
        "aggregate_only": True,
    },
    "outputs": {},
    "source_hashes": source_hashes.to_dict(orient="records"),
}
for output_path in (coverage_path, gate_path, source_hashes_path, figure_path):
    manifest["outputs"][output_path.name] = {
        "sha256": sha256_file(output_path),
        "bytes": output_path.stat().st_size,
    }
manifest_path.write_text(
    json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)

display(
    pd.DataFrame(
        [
            {"artifact": path.name, "sha256": sha256_file(path)}
            for path in (coverage_path, gate_path, source_hashes_path, figure_path, manifest_path)
        ]
    )
)

## Interpretation

This batch is useful because it starts the only genuinely prospective evidence
stream available to the retained LSEG/Gemma strategy. It is not yet useful for
judging alpha. The closed interval can contain fewer than one quarter of the
frozen minimum calendar sessions, so calculating active dates, scoring new
headlines, or opening returns would add cost and researcher degrees of freedom
without permitting the preregistered test. Accumulate immutable batches through
at least the 120th possible XNYS session, then apply the remaining active-session
gates before the one-shot prospective replay.